## Il problema è di prevedere il prezzo di una casa date alcune sue caratteristiche.

Il vostro primo obiettivo è quello di arrivare a compiere l’operazione di previsione nel modo più rapido possibile: 
- non concentratevi sull’accuratezza o sul fare tutti gli step possibili nella fase di pulizia dei dati ma sull’avere un codice “minimo” che almeno funzioni. In base al tempo a vostra disposizione sceglierete eventualmente altri aspetti su cui concentrarvi.

La descrizione delle variabili del dataset è disponibile presso: https://www.kaggle.com/datasets/yasserh/housing-prices-dataset

In [147]:
# Librerie
from pathlib import Path
import pandas as pd
import numpy as np
# gestione percorso
root = Path.cwd()
file_path = root / 'Housing.csv'

In [148]:
# Leggo il dataset
case_df = pd.read_csv(file_path)
case_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   price             545 non-null    int64
 1   area              545 non-null    int64
 2   bedrooms          545 non-null    int64
 3   bathrooms         545 non-null    int64
 4   stories           545 non-null    int64
 5   mainroad          545 non-null    str  
 6   guestroom         545 non-null    str  
 7   basement          545 non-null    str  
 8   hotwaterheating   545 non-null    str  
 9   airconditioning   545 non-null    str  
 10  parking           545 non-null    int64
 11  prefarea          545 non-null    str  
 12  furnishingstatus  545 non-null    str  
dtypes: int64(6), str(7)
memory usage: 55.5 KB


In [149]:
# visualizzo prime 5 righe del dataset
case_df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [150]:
# Mostra tutte le righe che sono considerate duplicati
is_duplicate = case_df[case_df.duplicated()]
print(is_duplicate)

Empty DataFrame
Columns: [price, area, bedrooms, bathrooms, stories, mainroad, guestroom, basement, hotwaterheating, airconditioning, parking, prefarea, furnishingstatus]
Index: []


In [151]:
# Filtra il DataFrame mostrando solo le righe con almeno un NaN
is_null = case_df[case_df.isnull().any(axis=1)]
print(is_null)

Empty DataFrame
Columns: [price, area, bedrooms, bathrooms, stories, mainroad, guestroom, basement, hotwaterheating, airconditioning, parking, prefarea, furnishingstatus]
Index: []


*dopo una prima analisi:*
- non riscontro duplicati
- non riscontro valori null

## Encoding (nello specifico Binary Encoding) 
Procedo con trasformare variabili categoriche testuali in valori numerici in modo tale che il modello possa elaborarli.
- `mainroad`	
- `guestroom`	
- `basement`	
- `hotwaterheating`	
- `airconditioning`
- `prefarea`
- e in fine `furnishingstatus`

In [152]:
cols_binarie = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
# Verifico e ottengo un set di tutti i valori presenti nelle colonne interessate
valori_unici = set(case_df[cols_binarie].values.ravel())
print(f"Valori presenti nelle colonne binarie: {valori_unici}\n")

# Verifica specifica per la categorica multipla della colonna 'furnishingstatus' uso value_counts()
# anche per capire se una categoria è rarissima
valori_cat_unici =case_df['furnishingstatus'].value_counts()
print(f"Valori presenti nella colonna 'furnishingstatus':\n{valori_cat_unici}")

Valori presenti nelle colonne binarie: {'yes', 'no'}

Valori presenti nella colonna 'furnishingstatus':
furnishingstatus
semi-furnished    227
unfurnished       178
furnished         140
Name: count, dtype: int64


In [153]:
# provvediamo a fare gli encoding binari
mapping = {'yes': 1, 'no': 0}

for col in cols_binarie:
    case_df[col] = case_df[col].map(mapping)

# Verifica dell'encoding
print(case_df[cols_binarie].head())

   mainroad  guestroom  basement  hotwaterheating  airconditioning  prefarea
0         1          0         0                0                1         1
1         1          0         0                0                1         0
2         1          0         1                0                0         1
3         1          0         1                0                1         1
4         1          1         1                0                1         0


In [ ]:
"""
# Trasformazione One-Hot Encoding (Variabili Dummy)
case_df = pd.get_dummies(case_df, columns=['furnishingstatus'], drop_first=True, dtype=int)

# Verifica le nuove colonne create
print(case_df.columns)

# Mostra le prime righe delle nuove colonne dummy
print(case_df.filter(like='furnishingstatus').head())
"""

"\n# Trasformazione One-Hot Encoding (Variabili Dummy)\ncase_df = pd.get_dummies(case_df, columns=['furnishingstatus'], drop_first=True, dtype=int)\n\n# Verifica le nuove colonne create\nprint(case_df.columns) \n\n# Mostra le prime righe delle nuove colonne dummy\nprint(case_df.filter(like='furnishingstatus').head())\n"

In [155]:
# Creiamo il dizionario di mappatura per la colonna 'furnishingstatus'
cat_mapping = {
    'unfurnished': 0,
    'semi-furnished': 1,
    'furnished': 2
}

# Applichiamo il mapping
case_df['furnishingstatus'] = case_df['furnishingstatus'].map(cat_mapping)

# Verifica del mapping
print(case_df['furnishingstatus'].unique())

[2 1 0]


In [156]:
case_df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,2
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,2
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,1
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,2
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,2
